# 基于一阶马尔可夫链与 FPMC 的 MovieLens-1M 下一项电影推荐实验

本 Notebook 基于当前项目代码组织一个可顺序运行的实验，比较五种 next-item recommendation 方法：

| Model | Score | 说明 |
| --- | --- | --- |
| Popularity | `score(j)=count(j)` | 非序列热门 baseline |
| Markov Chain | `score(j)=P(j | i_t)` | 显式一阶马尔可夫转移矩阵 |
| MF-only | `<p_u, q_j>` | 只建模用户长期偏好 |
| FMC-only | `<a_i, b_j>` | 因子分解的一阶 Markov 转移 |
| FPMC | `<p_u,q_j> + <a_i,b_j>` | 用户长期偏好 + 短期 Markov 转移 |

普通 Markov Chain 显式统计 `item_i -> item_j` 的转移概率。FMC 用低维向量点积近似 `item_i -> item_j` 的转移分数。FPMC 进一步加入 user-item matrix factorization，使推荐同时考虑用户长期偏好和短期状态转移。

In [ ]:
import importlib.util
import json
import math
import os
import random
import sys
import urllib.request
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

required_packages = ["matplotlib", "numpy", "torch", "tqdm"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing_packages:
    raise ImportError(
        "Missing packages: " + ", ".join(missing_packages) + ".\n"
        "Install them in this notebook kernel, for example:\n"
        "%pip install torch numpy matplotlib tqdm"
    )

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models.py").exists() and (PROJECT_ROOT / "Factorized_Personalized_Markov_Chains" / "models.py").exists():
    PROJECT_ROOT = PROJECT_ROOT / "Factorized_Personalized_Markov_Chains"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dataset import BPR_Dataset
from metrics import BPR_Loss
from models import FPMC
from preprocessing import create_df, create_user_history, create_user_noclick, reset_df, train_val_test_split

CONFIG = {
    "seed": 42,
    "min_user_len": 5,
    "num_neg_eval": 100,
    "batch_size": 1024,
    "epochs": 10,
    "lr": 1e-3,
    "k_ui": 64,
    "k_il": 64,
    "weight_decay": 1e-6,
    "num_workers": 0,
    "max_train_samples": 100_000,
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
print(torch.__version__)

## 下载 / 定位 MovieLens-1M

数据文件应位于 `ml-1m/ratings.dat`。如果不存在，将自动从 GroupLens 下载并解压。

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "ml-1m"
RATINGS_PATH = DATA_DIR / "ratings.dat"
ZIP_PATH = PROJECT_ROOT / "ml-1m.zip"
DATA_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

if not RATINGS_PATH.exists():
    print("ratings.dat not found, downloading MovieLens-1M...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(PROJECT_ROOT)
else:
    print(f"Found {RATINGS_PATH}")

assert RATINGS_PATH.exists(), "MovieLens-1M ratings.dat was not found or downloaded."

## 读取数据与 user/item 重编码

复用项目中的 `create_df` 和 `reset_df`。所有评分记录都作为交互行为，不按 rating 过滤。

In [ ]:
ratings = create_df("ml-1m/ratings.dat")
encoder = reset_df()
ratings = encoder.fit_transform(ratings)

n_users = ratings["user_id"].nunique()
n_items = ratings["item_id"].nunique()
print(f"users={n_users:,}, items={n_items:,}, interactions={len(ratings):,}")
ratings.head()

## 时间序列划分

使用 leave-one-out sequential split。每个用户按时间得到 `i1 -> i2 -> ... -> in`，训练集使用除最后两个转移外的全部转移，验证集使用倒数第二个转移，测试集使用最后一个转移。样本形式为 `(u, prev_item, next_item)`。

In [ ]:
user_history_all = create_user_history(ratings)
user_history = {u: hist for u, hist in user_history_all.items() if len(hist) >= CONFIG["min_user_len"]}
train_samples, val_samples, test_samples = train_val_test_split(user_history)

print(f"eligible users={len(user_history):,}")
print(f"train={len(train_samples):,}, val={len(val_samples):,}, test={len(test_samples):,}")
train_samples[:3], val_samples[:3], test_samples[:3]

## 构造 user_seen 与固定评估候选集

sampled ranking 对每个验证/测试样本使用同一套固定候选集：`[真实 next item] + 100 个负样本`。后续所有模型都在完全相同的候选集合上评估。

In [ ]:
user_seen = {u: set(hist) for u, hist in user_history.items()}
user_noclick = create_user_noclick(user_history, ratings, n_items)

def build_eval_candidates(samples, user_noclick, num_neg, seed):
    rng = np.random.default_rng(seed)
    candidate_rows = []
    for u, prev_i, true_i in samples:
        negatives, probabilities = user_noclick[u]
        replace = len(negatives) < num_neg
        sampled = rng.choice(negatives, size=num_neg, replace=replace, p=probabilities).tolist()
        candidate_rows.append([true_i] + sampled)
    return np.asarray(candidate_rows, dtype=np.int64)

val_candidates = build_eval_candidates(val_samples, user_noclick, CONFIG["num_neg_eval"], CONFIG["seed"] + 1)
test_candidates = build_eval_candidates(test_samples, user_noclick, CONFIG["num_neg_eval"], CONFIG["seed"] + 2)
print(val_candidates.shape, test_candidates.shape)

## Popularity baseline

Popularity 不使用用户或上一项，只按训练集中 item 出现次数打分：`score(j)=count(j)`。

In [ ]:
item_popularity = np.zeros(n_items, dtype=np.float32)
for _, _, next_i in train_samples:
    item_popularity[next_i] += 1.0

def score_popularity(samples, candidates):
    return item_popularity[candidates]

print("Top popular encoded item ids:", np.argsort(-item_popularity)[:10].tolist())

## Markov Chain baseline

显式统计训练集中 `prev_item -> next_item` 的一阶转移概率：`score(j)=P(j | i_t)`。如果某个上一项没有观测到转移，则退回到 popularity 分数。

In [ ]:
transition_counts = defaultdict(Counter)
transition_totals = Counter()
for _, prev_i, next_i in train_samples:
    transition_counts[prev_i][next_i] += 1
    transition_totals[prev_i] += 1

def score_markov(samples, candidates):
    scores = np.zeros_like(candidates, dtype=np.float32)
    for row_idx, (_, prev_i, _) in enumerate(samples):
        total = transition_totals[prev_i]
        if total == 0:
            scores[row_idx] = item_popularity[candidates[row_idx]]
            continue
        row_counts = transition_counts[prev_i]
        scores[row_idx] = [row_counts[int(item)] / total for item in candidates[row_idx]]
    return scores

print(f"observed previous items with transitions={len(transition_counts):,}")

## FPMC 模型定义

本项目的 `models.FPMC` 已支持三种模式：

- `mode="mf"`: `s_MF(u,j)=<p_u,q_j>`
- `mode="fmc"`: `s_FMC(i,j)=<a_i,b_j>`
- `mode="full"`: `s_FPMC(u,i,j)=s_MF(u,j)+s_FMC(i,j)`

In [ ]:
for mode in ["mf", "fmc", "full"]:
    probe_model = FPMC(n_users, n_items, CONFIG["k_ui"], CONFIG["k_il"], mode=mode)
    probe_out = probe_model(torch.tensor([0]), torch.tensor([0]), torch.tensor([0]))
    print(mode, tuple(probe_out.shape))
del probe_model, probe_out

## 训练函数与评估函数

训练使用项目中的 `BPR_Dataset` 和 `BPR_Loss`。BPR pairwise loss 为：

```python
F.softplus(-(pos_scores - neg_scores)).mean()
```

评估指标包括 `Hit@5`、`Hit@10`、`MRR@10`、`NDCG@10`。候选集中真实 item 固定在第 0 列。

In [ ]:
def bpr_collate_fn(batch):
    uid, prev_i, pos_i, neg_i = zip(*batch)
    return (
        torch.tensor(uid, dtype=torch.long),
        torch.tensor(prev_i, dtype=torch.long),
        torch.tensor(pos_i, dtype=torch.long),
        torch.tensor(neg_i, dtype=torch.long),
    )

def make_train_loader(samples):
    dataset = BPR_Dataset(samples, n_users, n_items)
    if CONFIG["max_train_samples"] is not None and len(dataset) > CONFIG["max_train_samples"]:
        rng = np.random.default_rng(CONFIG["seed"])
        indices = rng.choice(len(dataset), size=CONFIG["max_train_samples"], replace=False)
        dataset = Subset(dataset, indices.tolist())
        print(f"Using {len(dataset):,} sampled training transitions for smoke test.")
    return DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], collate_fn=bpr_collate_fn)

def train_torch_model(mode):
    set_seed(CONFIG["seed"])
    model = FPMC(n_users, n_items, CONFIG["k_ui"], CONFIG["k_il"], mode=mode).to(device)
    criterion = BPR_Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    train_loader = make_train_loader(train_samples)
    history = []

    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        losses = []
        for uid, prev_i, pos_i, neg_i in tqdm(train_loader, desc=f"{mode} epoch {epoch}", leave=False):
            uid = uid.to(device).long()
            prev_i = prev_i.to(device).long()
            pos_i = pos_i.to(device).long()
            neg_i = neg_i.to(device).long()

            optimizer.zero_grad()
            pos_scores = model(uid, prev_i, pos_i)
            neg_scores = model(uid, prev_i, neg_i)
            loss = criterion(pos_scores, neg_scores)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        mean_loss = float(np.mean(losses))
        history.append(mean_loss)
        print(f"{mode} epoch {epoch:02d}: loss={mean_loss:.4f}")

    return model, history

def score_torch_model(model, samples, candidates, batch_size=2048):
    model.eval()
    all_scores = []
    with torch.no_grad():
        for start in range(0, len(samples), batch_size):
            end = min(start + batch_size, len(samples))
            batch_samples = samples[start:end]
            batch_candidates = candidates[start:end]
            width = batch_candidates.shape[1]

            uid = torch.tensor([s[0] for s in batch_samples], dtype=torch.long, device=device).repeat_interleave(width)
            prev_i = torch.tensor([s[1] for s in batch_samples], dtype=torch.long, device=device).repeat_interleave(width)
            cand_i = torch.tensor(batch_candidates.reshape(-1), dtype=torch.long, device=device)

            scores = model(uid, prev_i, cand_i).reshape(end - start, width)
            all_scores.append(scores.cpu().numpy())
    return np.vstack(all_scores)

def ranking_metrics(scores):
    order = np.argsort(-scores, axis=1)
    true_positions = np.argwhere(order == 0)[:, 1] + 1
    hit5 = np.mean(true_positions <= 5)
    hit10 = np.mean(true_positions <= 10)
    mrr10 = np.mean(np.where(true_positions <= 10, 1.0 / true_positions, 0.0))
    ndcg10 = np.mean(np.where(true_positions <= 10, 1.0 / np.log2(true_positions + 1), 0.0))
    return {"Hit@5": float(hit5), "Hit@10": float(hit10), "MRR@10": float(mrr10), "NDCG@10": float(ndcg10)}

def evaluate_scores(name, scores):
    row = {"Model": name}
    row.update(ranking_metrics(scores))
    return row

MODEL_ORDER = ["Popularity", "Markov Chain", "MF-only", "FMC-only", "FPMC"]
METRIC_COLUMNS = ["Hit@5", "Hit@10", "MRR@10", "NDCG@10"]

def ordered_results(rows):
    by_model = {row["Model"]: row for row in rows}
    return [by_model[name] for name in MODEL_ORDER if name in by_model]

def display_results(rows):
    rows = ordered_results(rows)
    header = "| Model | Hit@5 | Hit@10 | MRR@10 | NDCG@10 |\n| --- | ---: | ---: | ---: | ---: |"
    body = [
        f"| {row['Model']} | {row['Hit@5']:.4f} | {row['Hit@10']:.4f} | {row['MRR@10']:.4f} | {row['NDCG@10']:.4f} |"
        for row in rows
    ]
    table = header + "\n" + "\n".join(body)
    try:
        from IPython.display import Markdown, display
        display(Markdown(table))
    except Exception:
        print(table)
    return rows

results = []
results.append(evaluate_scores("Popularity", score_popularity(test_samples, test_candidates)))
results.append(evaluate_scores("Markov Chain", score_markov(test_samples, test_candidates)))
display_results(results)

## 训练 MF-only

In [ ]:
mf_model, mf_loss = train_torch_model("mf")
results.append(evaluate_scores("MF-only", score_torch_model(mf_model, test_samples, test_candidates)))
display_results(results)

## 训练 FMC-only

In [ ]:
fmc_model, fmc_loss = train_torch_model("fmc")
results.append(evaluate_scores("FMC-only", score_torch_model(fmc_model, test_samples, test_candidates)))
display_results(results)

## 训练 FPMC

In [ ]:
fpmc_model, fpmc_loss = train_torch_model("full")
results.append(evaluate_scores("FPMC", score_torch_model(fpmc_model, test_samples, test_candidates)))
display_results(results)

## 结果汇总表

最终结果保存到：

- `results/fpmc_markov_results.csv`
- `results/fpmc_markov_results.json`

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)

import csv

results_table = display_results(results)

with open(results_dir / "fpmc_markov_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Model"] + METRIC_COLUMNS)
    writer.writeheader()
    writer.writerows(results_table)

with open(results_dir / "fpmc_markov_results.json", "w") as f:
    json.dump({row["Model"]: {metric: row[metric] for metric in METRIC_COLUMNS} for row in results_table}, f, indent=2)

print(results_dir / "fpmc_markov_results.csv")
print(results_dir / "fpmc_markov_results.json")

## 可视化

In [ ]:
models = [row["Model"] for row in results_table]
x = np.arange(len(models))
width = 0.18

fig, ax = plt.subplots(figsize=(10, 4))
for offset, metric in enumerate(METRIC_COLUMNS):
    values = [row[metric] for row in results_table]
    ax.bar(x + (offset - 1.5) * width, values, width, label=metric)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha="right")
ax.set_ylabel("score")
ax.set_title("Sampled ranking metrics on MovieLens-1M")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(mf_loss, label="MF-only")
plt.plot(fmc_loss, label="FMC-only")
plt.plot(fpmc_loss, label="FPMC")
plt.xlabel("epoch")
plt.ylabel("BPR loss")
plt.title("Training loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 实验结论

本实验用 sampled ranking 比较了非序列 popularity、显式一阶 Markov Chain，以及三个基于向量因子分解的 PyTorch 模型。Popularity 反映全局热门偏置；显式 Markov Chain 只使用上一部电影到下一部电影的统计转移；MF-only 只使用用户长期偏好；FMC-only 用低维向量表示短期转移；FPMC 将长期偏好与短期转移相加。

默认配置为了便于 CPU smoke test，会抽样最多 `CONFIG["max_train_samples"]` 条训练转移并训练 10 个 epoch。若需要更稳定的实验结果，可以把 `CONFIG["max_train_samples"]` 改为 `None`，并把 `CONFIG["epochs"]` 提高到 30 或更多。MovieLens 评分序列并不一定具有很强的短期转移模式，因此 FPMC 不显著优于 FMC-only 或 MF-only 并不必然表示实现失败，需要结合训练轮数、负采样规模和候选评估协议一起判断。